In [ ]:
import numpy as np

from pymatgen.core import Structure, Composition, Element, Lattice
from pymatgen.io.cif import CifWriter
from chggen.common.data_utils import mkdir


import argparse
import json
import pandas as pd
import os
import time
from datetime import datetime

In [ ]:
# def get_file_list(dir, key = 'csv'):
#     return [f for f in os.listdir(dir) if f.endswith(key)]

# ROOT_dir = './files/gen_structures_MV_new/'
# ROOT_dir = './files/gen_structures_NaTaCl/'

# chemical_list = get_file_list(ROOT_dir, key='')
# print(chemical_list)

# df_all = pd.DataFrame()
# for chemical_formula in chemical_list:
#     ROOT = ROOT_dir + chemical_formula + '/'
#     file_list = get_file_list(ROOT)

#     df = pd.read_csv(ROOT + file_list[0])

#     df_all = pd.concat([df_all, df], ignore_index=True)

# df_all

In [ ]:
df_all = pd.read_csv('./files/genSummary_merged_data.csv')


In [ ]:
df_all

In [ ]:
df_all = df_all.sort_values(['formula', 'e_hull_chgnet'])
df_all['min_e_hull'] = df_all.groupby('formula')['e_hull_chgnet'].transform('min')
df_all['filtered'] = (df_all['e_hull_chgnet'] <= 0.03) | (df_all['e_hull_chgnet'] == df_all['min_e_hull'])
df_all = df_all[df_all['filtered']].drop(['min_e_hull', 'filtered'], axis=1)


df_all = df_all.sort_values('e_hull_chgnet')
df_all['e_hull_diff'] = df_all.groupby(['formula', 'spacegroup_refine'])['e_hull_chgnet'].diff()
# df_all
df_all = df_all[(df_all['e_hull_diff'].isna()) | (df_all['e_hull_diff'].abs() >= 0.01)]

df_all = df_all.drop('e_hull_diff', axis=1)


In [ ]:
print(len(df_all))

In [ ]:
save_root = './forVASP_Na-M-Cl_' + datetime.now().strftime('%Y-%m-%d')

for idx, row in df_all.iterrows():
    structure = Structure.from_str(row['s_relax_refine_cif'], fmt='cif')
    save_json = {'material_id': row['material_id'],
                 'e_hull_chgnet': row['e_hull_chgnet'],
                'structure': structure.as_dict(),}

    mkdir(save_root + '/INPUT_' + row['material_id'])


    with open(save_root + '/INPUT_' + row['material_id'] + '/s.json', 'w') as fp:
        json.dump(save_json, fp, indent=4)


df_all.to_csv(save_root + '/generate_summary.csv', index=False)

In [ ]:
df_all